In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

¡Diagnóstico completado! El resultado del HTML nos confirma exactamente lo que sospechábamos: el sitio tiene una capa de optimización (LiteSpeed) y carga dinámica que bloquea las peticiones simples de requests.

Cuando Python leyó la página, solo vio el "esqueleto" (por eso solo encontró el título del "Carrito"), pero el contenido de las cervezas aún no se había "dibujado" en la pantalla. Para Python, la estantería estaba vacía.

La Solución: Subir de nivel a Selenium
Como mencionamos antes, Selenium es la herramienta que usan los profesionales para estos casos. Vamos a engañar al sitio abriendo un navegador real (Chrome) que espere a que las cervezas aparezcan.



In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://cultocervecero.com/categoria-producto/cervezas/cervecerias/"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# Imprime un pedazo del HTML para ver qué etiquetas hay
print("--- INICIO DEL HTML RECIBIDO ---")
print(response.text[:1000]) 
print("--- FIN DEL HTML ---")

# Intento de encontrar CUALQUIER h2 (donde suelen ir los nombres)
titulos = soup.find_all('h2')
print(f"\nSe encontraron {len(titulos)} etiquetas H2. Aquí las primeras 3:")
for t in titulos[:3]:
    print(f"- {t.text.strip()}")

--- INICIO DEL HTML RECIBIDO ---
<!DOCTYPE html><html lang="es" prefix="og: https://ogp.me/ns#"><head><script data-no-optimize="1">var litespeed_docref=sessionStorage.getItem("litespeed_docref");litespeed_docref&&(Object.defineProperty(document,"referrer",{get:function(){return litespeed_docref}}),sessionStorage.removeItem("litespeed_docref"));</script> <meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no" /><link rel="profile" href="http://gmpg.org/xfn/11"><link rel="pingback" href="https://cultocervecero.com/xmlrpc.php"><style>img:is([sizes="auto" i], [sizes^="auto," i]) { contain-intrinsic-size: 3000px 1500px }</style> <script data-cfasync="false" data-pagespeed-no-defer>var gtm4wp_datalayer_name = "dataLayer";
	var dataLayer = dataLayer || [];
	const gtm4wp_use_sku_instead = 0;
	const gtm4wp_currency = 'COP';
	const gtm4wp_product_per_impression = 10;
	const gtm4wp_clear_ecommerce = false;
	const gtm4wp_data

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

In [4]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC



In [19]:


# 1. Configuración Inicial 🌐
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 20)

try:
    driver.get("https://thecapitalbeer.com/shop-list/")

    # 2. Muro de Edad 🔞
    boton_si = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(., 'Si')]")))
    driver.execute_script("arguments[0].click();", boton_si)
    time.sleep(2)

    # 3. Scroll Infinito para cargar todo el catálogo 🖱️
    ultima_altura = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(4)
        nueva_altura = driver.execute_script("return document.body.scrollHeight")
        if nueva_altura == ultima_altura:
            break
        ultima_altura = nueva_altura

    # 4. Capturar Nombre, Precio y URL (Antes de entrar al enlace) 🔗
    bloques = driver.find_elements(By.CLASS_NAME, "qodef-e-product-content")
    lista_productos = []

    for b in bloques:
        try:
            nombre = b.find_element(By.CLASS_NAME, "qodef-e-product-title").text
            precio = b.find_element(By.CLASS_NAME, "price").text
            link = b.find_element(By.TAG_NAME, "a").get_attribute("href")
            
            lista_productos.append({
                "Nombre": nombre,
                "Precio": precio,
                "URL": link
            })
        except:
            continue # Si falta algún dato básico, saltamos ese producto

    # 5. Navegación y Extracción de Ficha Técnica 🔍
    datos_completos = []
    claves_interes = ["País", "Estilo", "IBU", "Productor", "Fermentación", "Graduación"]

    for producto in lista_productos:
        driver.get(producto["URL"])
        
        try:
            contenedor = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "woocommerce-product-details__short-description")))
            lineas = contenedor.text.split('\n')
            
            # Creamos una copia de los datos básicos (Nombre, Precio, URL)
            ficha_tecnica = producto.copy()
            
            for linea in lineas:
                if any(linea.startswith(clave) for clave in claves_interes) and ":" in linea:
                    etiqueta, valor = linea.split(":", 1)
                    ficha_tecnica[etiqueta.strip()] = valor.strip()
            
            datos_completos.append(ficha_tecnica)
            print(f"✅ Procesada: {ficha_tecnica['Nombre']}")
        except Exception as e:
            print(f"❌ Error en {producto['URL']}: {e}")

    # 6. Guardar en CSV 📊
    df = pd.DataFrame(datos_completos)
    df.to_csv("cervezas_portafolio.csv", index=False)
    print("✨ ¡Proceso finalizado con éxito!")

finally:
    driver.quit()

✅ Procesada: CERVEZA ALEMANA AC DC LATA 568 ML
✅ Procesada: CERVEZA ALEMANA BENEDIKTINER WEISSBIER BOTELLA 330 ML
✅ Procesada: CERVEZA ALEMANA BENEDIKTINER WEISSBIER BOTELLA 500 ML
✅ Procesada: CERVEZA ALEMANA BENEDIKTINER WEISSBIER LATA 500 ML
✅ Procesada: CERVEZA ALEMANA BENEDIKTINER WEISSBIER BARRIL 5 LITROS
✅ Procesada: CERVEZA ALEMANA BITBURGER BOTELLA 330 ML
✅ Procesada: CERVEZA ALEMANA BITBURGER BOTELLA 500 ML
✅ Procesada: CERVEZA ALEMANA BITBURGER LATA 500 ML
✅ Procesada: ESTUCHE DE CERVEZA BELGA CHIMAY
✅ Procesada: ESTUCHE DE CERVEZA BELGA DELIRIUM
✅ Procesada: ESTUCHE DE CERVEZA BELGA GULDEN DRAAK
✅ Procesada: ESTUCHE DE CERVEZA ESCOCESA BREWDOG
✅ Procesada: CERVEZA ALEMANA BENEDIKTINER WEISSBIER BARRIL 5 LITROS
✅ Procesada: CERVEZA ALEMANA BITBURGER BOTELLA 330 ML
✅ Procesada: CERVEZA ALEMANA BITBURGER BOTELLA 500 ML
✅ Procesada: CERVEZA ALEMANA BITBURGER LATA 500 ML
✅ Procesada: CERVEZA ALEMANA BITBURGER BARRIL 5 LITROS
✅ Procesada: CERVEZA ALEMANA BITBURGER DRIVE BOTELLA 3

In [22]:
cervezas = pd.read_csv("cervezas_portafolio.csv")
print(cervezas.head())

                                              Nombre     Precio  \
0                  CERVEZA ALEMANA AC DC LATA 568 ML   $ 16.500   
1  CERVEZA ALEMANA BENEDIKTINER WEISSBIER BOTELLA...   $ 11.000   
2  CERVEZA ALEMANA BENEDIKTINER WEISSBIER BOTELLA...   $ 14.000   
3  CERVEZA ALEMANA BENEDIKTINER WEISSBIER LATA 50...   $ 12.000   
4  CERVEZA ALEMANA BENEDIKTINER WEISSBIER BARRIL ...  $ 125.000   

                                                 URL          País de origen  \
0  https://thecapitalbeer.com/product/cerveza-ale...                Alemania   
1  https://thecapitalbeer.com/product/cerveza-ale...  Ettay Abbey – Alemania   
2  https://thecapitalbeer.com/product/cerveza-ale...  Ettay Abbey – Alemania   
3  https://thecapitalbeer.com/product/cerveza-ale...  Ettay Abbey – Alemania   
4  https://thecapitalbeer.com/product/cerveza-ale...  Ettay Abbey – Alemania   

                     Productor Fermentación     Estilo Graduación   IBU  
0           Karlsberg Brauerei        Lage

In [23]:
cervezas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Nombre          42 non-null     object 
 1   Precio          41 non-null     object 
 2   URL             42 non-null     object 
 3   País de origen  32 non-null     object 
 4   Productor       34 non-null     object 
 5   Fermentación    35 non-null     object 
 6   Estilo          35 non-null     object 
 7   Graduación      35 non-null     object 
 8   IBU             33 non-null     float64
dtypes: float64(1), object(8)
memory usage: 3.1+ KB


In [25]:
print(cervezas)

                                               Nombre     Precio  \
0                   CERVEZA ALEMANA AC DC LATA 568 ML   $ 16.500   
1   CERVEZA ALEMANA BENEDIKTINER WEISSBIER BOTELLA...   $ 11.000   
2   CERVEZA ALEMANA BENEDIKTINER WEISSBIER BOTELLA...   $ 14.000   
3   CERVEZA ALEMANA BENEDIKTINER WEISSBIER LATA 50...   $ 12.000   
4   CERVEZA ALEMANA BENEDIKTINER WEISSBIER BARRIL ...  $ 125.000   
5            CERVEZA ALEMANA BITBURGER BOTELLA 330 ML    $ 9.000   
6            CERVEZA ALEMANA BITBURGER BOTELLA 500 ML   $ 13.000   
7               CERVEZA ALEMANA BITBURGER LATA 500 ML   $ 12.000   
8                     ESTUCHE DE CERVEZA BELGA CHIMAY   $ 99.000   
9                   ESTUCHE DE CERVEZA BELGA DELIRIUM  $ 110.000   
10              ESTUCHE DE CERVEZA BELGA GULDEN DRAAK   $ 70.000   
11                ESTUCHE DE CERVEZA ESCOCESA BREWDOG   $ 55.000   
12  CERVEZA ALEMANA BENEDIKTINER WEISSBIER BARRIL ...  $ 125.000   
13           CERVEZA ALEMANA BITBURGER BOTELLA 3

In [26]:
# Guardar como Excel
# index=False evita que se cree una columna extra con los números de fila
df.to_excel("cervezas_portafolio.xlsx", index=False)

print("📊 ¡Archivo Excel generado con éxito!")

📊 ¡Archivo Excel generado con éxito!
